# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 9 — Attribution across models, and the missing cell of the design

Three claims in the manuscript rest on a single model: the comparison between attribution
methods, the behaviour of a network trained on permuted labels, and the reading of
GNNExplainer as falling below chance. A paper whose argument is that one draw is not a
distribution cannot leave them there.

This notebook trains twenty graph models and repeats every attribution measurement across
all of them, runs the permutation control twenty times in each family, and fills the cell of
the design that separates the attribution rule from the molecular representation.

**Every intermediate artefact is saved**: model weights, raw atom-level attribution scores at
five retained fractions, and the test performance of each model. The previous round kept only
one summary statistic, which is why these models have to be trained again.

**Contents**

1. Environment and shared definitions
2. Data
3. Twenty trained models, with weights and scores retained
4. Attribution across the twenty models
5. Does agreement track predictive performance
6. Agreement over the course of training
7. The permutation control, twenty times in each family
8. The missing cell: gradient attribution on fingerprints
9. GNNExplainer under different settings
10. Sensitivity of the untrained distribution to architecture size
11. Applicability domain and conformal coverage for the descriptor model
12. Verification of the precision-recall computation
13. Summary

*Approximate runtime: six to seven hours on a T4. Each stage checkpoints, so an interrupted
session resumes where it stopped.*

## 1. Environment and shared definitions

In [ ]:
!pip install -q rdkit
!pip install -q xgboost
!pip install -q shap
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 57.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain", f"{WORK}/models"
CKPT = f"{WORK}/ckpt_part9"
for d in (R, EXPL, MODELS, CKPT): os.makedirs(d, exist_ok=True)
print("checkpoints:", CKPT)

Mounted at /content/drive
checkpoints: /content/drive/MyDrive/ames_pipeline/ckpt_part9


In [2]:
%%writefile /content/ames9.py
"""Definitions shared across Part 9, matching the protocol used in Parts 2 to 5."""
import random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator as rfg
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*"); warnings.filterwarnings("ignore")
FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']
morgan_gen = rfg.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr

def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None: X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)

def one_hot(x, allowable):
    if x not in allowable: x = allowable[-1]
    return [int(x == a) for a in allowable]

def atom_features(a):
    f  = one_hot(a.GetSymbol(), ATOM_LIST) + one_hot(str(a.GetHybridization()), HYBRIDIZATION)
    f += [a.GetDegree(), a.GetFormalCharge(), int(a.GetIsAromatic()),
          a.GetTotalNumHs(), int(a.IsInRing())]
    return f

def bond_features(b):
    bt = b.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(b.GetIsConjugated()), int(b.IsInRing())]

def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        ei += [[i, j], [j, i]]; ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)

def build_dataset(d):
    g = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [x for x in g if x is not None]

class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn": conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else: raise ValueError(kind)
            self.convs.append(conv); self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(x, ei, ea) if self.use_edge_attr else conv(x, ei)
            h = F.relu(bn(h)); h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, b), global_add_pool(x, b)], dim=1)
        return self.head(g).squeeze(-1)

def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4, use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads, use_edge_attr=use_edge_attr).to(DEVICE)

class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = x, edge_index, edge_attr, batch

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(DEVICE)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(model(d)).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, use_edge_attr=True, epoch_hook=None):
    """The protocol used for every model in this study."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"],
                           weight_decay=params["weight_decay"])
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    tl = DataLoader(tr_set, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    vl = DataLoader(va_set, batch_size=256)
    best, best_state, waited = -1.0, None, 0
    for ep in range(max_epochs):
        model.train()
        for d in tl:
            d = d.to(DEVICE); opt.zero_grad()
            crit(model(d), d.y.view(-1)).backward(); opt.step()
        y, p = evaluate(model, vl)
        auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
        if epoch_hook is not None: epoch_hook(ep, model, auc)
        if auc > best:
            best, waited = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience: break
    model.load_state_dict(best_state); model.eval()
    return model, best

def train_to_fit(kind, params, seed, dataset, n_epochs, dropout=0.0, weight_decay=0.0,
                 lr=None):
    """Fit a target with regularisation and early stopping switched off."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], dropout,
                      params.get("residual", True))
    opt = torch.optim.Adam(model.parameters(), lr=lr or params["lr"],
                           weight_decay=weight_decay)
    crit = nn.BCEWithLogitsLoss()
    ld = DataLoader(dataset, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    last = float("nan")
    for ep in range(n_epochs):
        model.train(); tot = 0.0
        for d in ld:
            d = d.to(DEVICE); opt.zero_grad()
            loss = crit(model(d), d.y.view(-1)); loss.backward(); opt.step()
            tot += loss.item()
        last = tot / max(len(ld), 1)
    model.eval(); return model, last

Overwriting /content/ames9.py


In [3]:
import shutil, os
shutil.copy("/content/ames9.py", "/content/drive/MyDrive/ames_pipeline/ames9.py")
print(os.path.exists("/content/drive/MyDrive/ames_pipeline/ames9.py"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ames_pipeline/ames9.py'

## 2. Data

In [ ]:
import sys, json, time, glob
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames9 import *
from sklearn.metrics import roc_auc_score, average_precision_score

SEED, PRIMARY_SPLIT_SEED = 42, 101
N_MODELS, N_PERM = 20, 14
MODEL_SEEDS = list(range(20000, 20000 + N_MODELS))
K_FRAC, K_GRID, N_IG_STEPS, N_TIEBREAK = 0.30, [0.10,0.20,0.30,0.40,0.50], 32, 200
HP_GNN     = json.load(open(f"{R}/hpo_gnn.json"))
HP_CLASSIC = json.load(open(f"{R}/hpo_classic.json"))
hp = HP_GNN["gin"]

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
df_train, df_valid, df_test = [x.reset_index(drop=True)
    for x in make_scaffold_split(df, seed=PRIMARY_SPLIT_SEED)]
ref = pd.read_csv(f"{R}/primary_test.csv")
assert set(df_test["canonical"]) == set(ref["canonical"]), "split mismatch"
print("primary partition reproduced")

X_train, df_train = build_X(df_train); y_train = df_train["label"].values
X_valid, df_valid = build_X(df_valid); y_valid = df_valid["label"].values
X_test,  df_test  = build_X(df_test);  y_test  = df_test["label"].values
train_set, valid_set, test_set = build_dataset(df_train), build_dataset(df_valid), build_dataset(df_test)
test_loader = DataLoader(test_set, batch_size=256)
POS_W = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)],
                     dtype=torch.float, device=DEVICE)
print(f"train {len(train_set)}, valid {len(valid_set)}, test {len(test_set)}; device {DEVICE}")

primary partition reproduced
train 5804, valid 725, test 726; device cuda


In [ ]:
TOXICOPHORES = {
    "aromatic_nitro":"[a][N+](=O)[O-]", "aromatic_amine":"[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
    "aromatic_nitroso":"[a][NX2]=[OX1]", "alkyl_nitrite":"[CX4][OX2][NX2]=[OX1]",
    "nitrosamine":"[NX3][NX2]=[OX1]", "epoxide":"C1OC1", "aziridine":"C1NC1",
    "azide":"[NX2]=[NX2+]=[NX1-]", "azo_aromatic":"[a][NX2]=[NX2][a]",
    "aliphatic_halide":"[CX4][F,Cl,Br,I]", "hetero_hetero_bond":"[N,O;H1,H2][N,O;H1,H2]",
    "polycyclic_aromatic":"c1ccc2ccccc2c1", "aromatic_hydroxylamine":"[a][NX3;H1][OX2;H1]",
    "michael_acceptor":"[CX3]=[CX3][CX3]=[OX1]", "aromatic_azoxy":"[a][NX2]=[NX3+]([O-])[a]",
    "nitrogen_mustard":"[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
    "aromatic_hydrazine":"[a][NX3;H1][NX3;H2]",
    "alkyl_sulfonate":"[CX4][OX2][SX4](=[OX1])(=[OX1])",
}
PATTERNS = {k: Chem.MolFromSmarts(v) for k, v in TOXICOPHORES.items()}
assert all(p is not None for p in PATTERNS.values())

EVAL = []
for smi, lab in zip(df_test["canonical"], y_test):
    if lab != 1: continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() == 0: continue
    tox = set()
    for p in PATTERNS.values():
        for m in mol.GetSubstructMatches(p): tox.update(m)
    if not tox or len(tox) == mol.GetNumAtoms(): continue
    EVAL.append({"smiles": smi, "mol": mol, "tox": tox, "n_atoms": mol.GetNumAtoms()})
GRAPHS = [smiles_to_data(e["smiles"], 1.0) for e in EVAL]
CHANCE = np.array([len(e["tox"]) / e["n_atoms"] for e in EVAL])
N_EVAL = len(EVAL)

def overlap(scores, k_frac=K_FRAC, tie_seed=0):
    rng = np.random.default_rng(tie_seed); out = np.full(N_EVAL, np.nan)
    for i, (e, sc) in enumerate(zip(EVAL, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"]) / k
    return out

def lift(ov):
    m = ~np.isnan(ov)
    return float(ov[m].mean() / CHANCE[m].mean()) if m.sum() else np.nan

def scored(scores, k_frac=K_FRAC, n=N_TIEBREAK, base=0):
    return np.array([lift(overlap(scores, k_frac, base+j)) for j in range(n)])

ORACLE = [np.array([1.0 if a in e["tox"] else 0.0 for a in range(e["n_atoms"])]) for e in EVAL]
CEILING = {k: lift(overlap(ORACLE, k, 0)) for k in K_GRID}
print(f"evaluation set {N_EVAL}, chance {CHANCE.mean():.4f}, ceiling(k=.30) {CEILING[0.30]:.4f}")

DEG = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in EVAL]
DEG_LIFT = {k: scored(DEG, k, base=4000).mean() for k in K_GRID}
print("degree baseline:", {k: round(v, 4) for k, v in DEG_LIFT.items()})

evaluation set 237, chance 0.4360, ceiling(k=.30) 1.9144
degree baseline: {0.1: np.float64(1.2001), 0.2: np.float64(1.2172), 0.3: np.float64(1.2136), 0.4: np.float64(1.1677), 0.5: np.float64(1.1079)}


## 3. Twenty trained models

Twenty models are fitted from independent seeds under the protocol used throughout. Weights
are written to disk, so that any attribution question asked later needs no retraining. The
test performance of each is recorded alongside, which Section 5 uses.

In [ ]:
def grad_scores(model, graphs, method="ig", n_steps=N_IG_STEPS, batch_size=64):
    """Attribution scores. The model's own gradient state is saved and restored, so this
    can be called on a model that is still being trained."""
    was_training = model.training
    saved_flags = [p.requires_grad for p in model.parameters()]
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        batch = batch.to(DEVICE); x0 = batch.x.clone()
        if method == "ig":
            total = torch.zeros_like(x0)
            for al in torch.linspace(1.0/n_steps, 1.0, n_steps):
                xa = (x0*al).detach().requires_grad_(True)
                lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
                total += torch.autograd.grad(lg.sum(), xa)[0]
            attr = x0 * (total/n_steps)
        else:
            xa = x0.clone().detach().requires_grad_(True)
            lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            g = torch.autograd.grad(lg.sum(), xa)[0]
            attr = g if method == "saliency" else x0*g
        sc = attr.abs().sum(dim=1).detach().cpu().numpy(); b = batch.batch.cpu().numpy()
        for gi in range(batch.num_graphs): out.append(sc[b == gi])
    for p, f in zip(model.parameters(), saved_flags): p.requires_grad_(f)
    model.train(was_training)
    return out

STATE = f"{CKPT}/models_meta.csv"
meta = pd.read_csv(STATE).to_dict("records") if os.path.exists(STATE) else []
done = {int(r["seed"]) for r in meta}
t0 = time.time()
for sd in MODEL_SEEDS:
    if sd in done: continue
    m, va = train_gnn("gin", hp, sd, train_set, valid_set, POS_W)
    torch.save({"state_dict": m.state_dict(), "valid_auc": float(va)},
               f"{MODELS}/gin_attr_seed{sd}.pt")
    y, p = evaluate(m, test_loader)
    rec = {"seed": sd, "valid_auc": va, "test_roc_auc": roc_auc_score(y, p),
           "test_pr_auc": average_precision_score(y, p)}
    for meth in ["ig", "gradinput", "saliency"]:
        sc = grad_scores(m, GRAPHS, meth)
        np.savez_compressed(f"{EXPL}/scores_{meth}_seed{sd}.npz",
                            **{f"m{i}": s for i, s in enumerate(sc)})
        for k in K_GRID:
            rec[f"lift_{meth}_k{int(k*100)}"] = scored(sc, k, n=50, base=sd).mean()
    meta.append(rec); pd.DataFrame(meta).to_csv(STATE, index=False)
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"  seed {sd}  valid {va:.4f}  test {rec['test_roc_auc']:.4f}  "
          f"IG lift {rec['lift_ig_k30']:.4f}   ({(time.time()-t0)/60:.1f} min)")

meta = pd.DataFrame(meta).sort_values("seed")
meta.to_csv(f"{R}/table42_model_level_attribution.csv", index=False)
print()
print(meta[["seed","valid_auc","test_roc_auc","lift_ig_k30"]].round(4).to_string(index=False))


 seed  valid_auc  test_roc_auc  lift_ig_k30
20000     0.8281        0.7864       1.2339
20001     0.8443        0.7894       1.3347
20002     0.8521        0.8013       1.3546
20003     0.8505        0.7931       1.2019
20004     0.8406        0.7860       1.2758
20005     0.8482        0.7774       0.9987
20006     0.8443        0.7882       1.1993
20007     0.8383        0.7845       1.2684
20008     0.8431        0.7857       1.2135
20009     0.8339        0.7870       1.2673
20010     0.8345        0.7584       1.2358
20011     0.8439        0.7843       1.2492
20012     0.8359        0.7766       1.2986
20013     0.8445        0.7800       1.2774
20014     0.8476        0.7763       1.3874
20015     0.8392        0.7866       1.4132
20016     0.8431        0.7953       1.3792
20017     0.8436        0.7794       1.1885
20018     0.8488        0.7938       1.2148
20019     0.8465        0.7893       1.3176


## 4. Attribution across the twenty models

Each measurement is now a distribution. The stored figures from the single representative
model are shown beside them, so that the effect of having used one model can be read
directly.

In [ ]:
from scipy.stats import wilcoxon, rankdata
rows = []
for meth, label in [("ig","Integrated Gradients"), ("gradinput","Gradient x Input"),
                    ("saliency","Saliency")]:
    for k in K_GRID:
        v = meta[f"lift_{meth}_k{int(k*100)}"].values
        rows.append({"method": label, "k_fraction": k, "n_models": len(v),
                     "lift_mean": v.mean(), "lift_sd": v.std(ddof=1),
                     "lift_min": v.min(), "lift_max": v.max(),
                     "degree_baseline": DEG_LIFT[k], "ceiling": CEILING[k],
                     "pct_exceeding_degree": 100*(v > DEG_LIFT[k]).mean(),
                     "pct_of_ceiling": 100*(v.mean()-1)/(CEILING[k]-1)})
across = pd.DataFrame(rows)
across.to_csv(f"{R}/table43_attribution_across_models.csv", index=False)
print(across[across.k_fraction == K_FRAC].round(4).to_string(index=False))
print()
v = meta["lift_ig_k30"].values
print(f"Integrated Gradients over {len(v)} models: {v.mean():.4f} +/- {v.std(ddof=1):.4f}, "
      f"range [{v.min():.4f}, {v.max():.4f}]")
print(f"degree baseline {DEG_LIFT[0.30]:.4f}; exceeded by "
      f"{100*(v > DEG_LIFT[0.30]).mean():.0f}% of models")
try:
    t17 = pd.read_csv(f"{R}/table17_topk_sensitivity.csv")
    rep = t17[(t17.source == "GIN (Integrated Gradients)") & (t17.k_fraction == 0.30)]
    if len(rep):
        print(f"single representative model reported earlier: {float(rep.lift.iloc[0]):.4f} "
              f"(percentile {100*(v < float(rep.lift.iloc[0])).mean():.0f} of this distribution)")
except FileNotFoundError:
    pass

              method  k_fraction  n_models  lift_mean  lift_sd  lift_min  lift_max  degree_baseline  ceiling  pct_exceeding_degree  pct_of_ceiling
Integrated Gradients         0.3        20     1.2655   0.0916    0.9987    1.4132           1.2136   1.9144                  75.0         29.0337
    Gradient x Input         0.3        20     1.1887   0.0911    1.0382    1.3351           1.2136   1.9144                  40.0         20.6375
            Saliency         0.3        20     1.2694   0.1010    1.0523    1.4244           1.2136   1.9144                  75.0         29.4656

Integrated Gradients over 20 models: 1.2655 +/- 0.0916, range [0.9987, 1.4132]
degree baseline 1.2136; exceeded by 75% of models
single representative model reported earlier: 1.2265 (percentile 30 of this distribution)


## 5. Does agreement track predictive performance

All twenty models were selected on validation ROC-AUC and fall in a narrow band. If agreement
with the alerts is a consequence of having learned the endpoint, it should rise with test
performance within that band. If it does not, then among models that have all learned the
task, the quality of the attribution is not a function of how much was learned.

In [ ]:
from scipy.stats import pearsonr, spearmanr
rows = []
for meth in ["ig","gradinput","saliency"]:
    x = meta["test_roc_auc"].values; y = meta[f"lift_{meth}_k30"].values
    pr, sp = pearsonr(x, y), spearmanr(x, y)
    rows.append({"method": meth, "n": len(x),
                 "roc_auc_min": x.min(), "roc_auc_max": x.max(),
                 "pearson_r": pr[0], "pearson_p": pr[1],
                 "spearman_r": sp.correlation, "spearman_p": sp.pvalue})
perf = pd.DataFrame(rows)
perf.to_csv(f"{R}/table44_lift_vs_performance.csv", index=False)
print(perf.round(4).to_string(index=False))
print()
print(f"test ROC-AUC spans {meta.test_roc_auc.min():.4f} to {meta.test_roc_auc.max():.4f} "
      f"across the twenty models")

   method  n  roc_auc_min  roc_auc_max  pearson_r  pearson_p  spearman_r  spearman_p
       ig 20       0.7584       0.8013     0.2557     0.2766      0.1789      0.4503
gradinput 20       0.7584       0.8013    -0.0040     0.9866     -0.0752      0.7527
 saliency 20       0.7584       0.8013    -0.2076     0.3797     -0.3248      0.1623

test ROC-AUC spans 0.7584 to 0.8013 across the twenty models


## 6. Agreement over the course of training

The share of the agreement present before training has been reported as a single ratio.
Measuring it at intervals during a single run turns that ratio into a curve, and shows whether
the change happens early, late, or gradually.

In [ ]:
EPOCH_PATH = f"{R}/table45_lift_by_epoch.csv"
if os.path.exists(EPOCH_PATH):
    ep_df = pd.read_csv(EPOCH_PATH); print("loaded from checkpoint")
else:
    marks = [0, 1, 2, 3, 5, 8, 12, 18, 25, 35, 50, 70, 100, 140, 190, 249]
    rec = []
    def hook(ep, model, auc):
        if ep in marks:
            sc = grad_scores(model, GRAPHS, "ig")
            rec.append({"epoch": ep+1, "valid_auc": auc,
                        "lift": scored(sc, n=50, base=9000+ep).mean()})
    set_seed(31337)
    m0 = build_gnn("gin", hp["hidden"], hp["n_layers"], hp["dropout"], hp.get("residual", True))
    sc0 = grad_scores(m0, GRAPHS, "ig")
    rec.append({"epoch": 0, "valid_auc": np.nan, "lift": scored(sc0, n=50, base=8999).mean()})
    del m0
    _m, _ = train_gnn("gin", hp, 31337, train_set, valid_set, POS_W, epoch_hook=hook)
    del _m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    ep_df = pd.DataFrame(rec).drop_duplicates("epoch").sort_values("epoch")
    ep_df["degree_baseline"] = DEG_LIFT[0.30]
    ep_df.to_csv(EPOCH_PATH, index=False)
print(ep_df.round(4).to_string(index=False))

loaded from checkpoint
 epoch  valid_auc   lift  degree_baseline
     0        NaN 1.1279           1.2136
     1     0.7364 1.3071           1.2136
     2     0.7331 1.2476           1.2136
     3     0.5841 1.2518           1.2136
     4     0.8090 1.2715           1.2136
     6     0.7945 1.2882           1.2136
     9     0.8196 1.2665           1.2136
    13     0.8348 1.2855           1.2136
    19     0.8199 1.2340           1.2136
    26     0.8118 1.3367           1.2136
    36     0.8354 1.2274           1.2136
    51     0.8437 1.2279           1.2136
    71     0.8249 1.0422           1.2136
   101     0.7969 1.1516           1.2136


## 7. The permutation control, twenty times in each family

A model fitted to randomised labels is only informative as a control if it fitted them.
Regularisation and early stopping are therefore switched off and the training performance on
the permuted target is recorded for every repeat, which also allows the agreement to be read
against how far the noise was absorbed.

In [ ]:
from xgboost import XGBClassifier
import shap
from rdkit.Chem import rdFingerprintGenerator as rfg2

def fp_bitinfo(mol):
    ao = rfg2.AdditionalOutput(); ao.AllocateBitInfoMap()
    morgan_gen.GetFingerprint(mol, additionalOutput=ao)
    return ao.GetBitInfoMap()

def bit_atoms(mol, ai, rad):
    if rad == 0: return {ai}
    env = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, ai); s = {ai}
    for b in env:
        bd = mol.GetBondWithIdx(b); s.add(bd.GetBeginAtomIdx()); s.add(bd.GetEndAtomIdx())
    return s

BITINFO = [fp_bitinfo(e["mol"]) for e in EVAL]
X_eval = np.array([fp_array(e["smiles"]) for e in EVAL])

def bits_to_atoms(vec_matrix):
    out = []
    for i, (e, bi) in enumerate(zip(EVAL, BITINFO)):
        vec = vec_matrix[i] if vec_matrix.ndim == 2 else vec_matrix
        s = np.zeros(e["n_atoms"])
        for bit, envs in bi.items():
            v = float(vec[bit])
            if v == 0.0 or len(envs) == 0: continue
            for ai, rad in envs:
                atoms = bit_atoms(e["mol"], ai, rad)
                share = v / (len(envs)*len(atoms))
                for a in atoms: s[a] += share
        out.append(s)
    return out

def tree_shap(model, X):
    v = shap.TreeExplainer(model).shap_values(X)
    if isinstance(v, list): v = v[-1]
    v = np.asarray(v)
    return v[:, :, -1] if v.ndim == 3 else v

PERM_PATH = f"{R}/table46_permutation_repeats.csv"
FIT_MIN_AUC, FIT_MIN_SD = 0.60, 1e-3

perm = pd.read_csv(PERM_PATH).to_dict("records") if os.path.exists(PERM_PATH) else []
# records written before the escalation logic may hold collapsed runs; redo those
kept = []
for r in perm:
    if r["family"] == "graph" and (r["train_auc_permuted"] < FIT_MIN_AUC
                                   or r.get("pred_sd", 1) < FIT_MIN_SD):
        continue
    kept.append(r)
if len(kept) < len(perm):
    print(f"discarding {len(perm)-len(kept)} collapsed graph runs from an earlier attempt")
perm = kept
done = {(r["family"], int(r["repeat"])) for r in perm}
y_orig = np.array([float(g.y.item()) for g in train_set])

def fit_permuted_graph(rep, train_perm):
    """Escalate until the network has actually fitted the randomised labels.

    A network that collapses to a constant output has not absorbed the noise, and its
    attributions are a random ranking rather than a control. Each attempt lowers the
    learning rate and lengthens the budget."""
    attempts = [(70_000+rep,      hp["lr"],        300),
                (170_000+rep,     hp["lr"]/3.0,    400),
                (270_000+rep,     hp["lr"]/10.0,   500)]
    for att, (sd, lr, ne) in enumerate(attempts, start=1):
        m, loss = train_to_fit("gin", hp, sd, train_perm, ne, lr=lr)
        ytr, ptr = evaluate(m, DataLoader(train_perm, batch_size=256))
        auc, sdev = roc_auc_score(ytr, ptr), float(ptr.std())
        if auc >= FIT_MIN_AUC and sdev >= FIT_MIN_SD:
            return m, loss, auc, sdev, att, True
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"    attempt {att} did not fit (train AUC {auc:.3f}, sd {sdev:.4f}); escalating")
    m, loss = train_to_fit("gin", hp, attempts[-1][0], train_perm, attempts[-1][2],
                           lr=attempts[-1][1])
    ytr, ptr = evaluate(m, DataLoader(train_perm, batch_size=256))
    return m, loss, roc_auc_score(ytr, ptr), float(ptr.std()), len(attempts), False

t0 = time.time()
for rep in range(N_PERM):
    rng = np.random.default_rng(70_000 + rep)
    if ("graph", rep) not in done:
        yp = rng.permutation(y_orig)
        tp = []
        for g, v in zip(train_set, yp):
            h = g.clone(); h.y = torch.tensor([v], dtype=torch.float); tp.append(h)
        m, loss, auc, sdev, att, ok = fit_permuted_graph(rep, tp)
        yte, pte = evaluate(m, test_loader)
        sc = grad_scores(m, GRAPHS, "ig")
        perm.append({"family":"graph","repeat":rep,
                     "train_auc_permuted":auc,"test_auc_true":roc_auc_score(yte,pte),
                     "pred_sd":sdev,"final_loss":loss,"attempts":att,"fitted":bool(ok),
                     "lift":scored(sc, n=50, base=70_000+rep).mean()})
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        pd.DataFrame(perm).to_csv(PERM_PATH, index=False)
        r_ = perm[-1]
        print(f"  graph {rep:2d}  train {r_['train_auc_permuted']:.3f}  "
              f"test {r_['test_auc_true']:.3f}  lift {r_['lift']:.4f}  "
              f"attempts {att}  fitted {ok}  ({(time.time()-t0)/60:.1f} min)")
    if ("tree", rep) not in done:
        ytp = rng.permutation(y_train)
        cfg = {k: v for k, v in HP_CLASSIC["XGBoost"].items() if k not in ("reg_alpha","reg_lambda")}
        xm = XGBClassifier(n_estimators=1500, **cfg, reg_alpha=0.0, reg_lambda=0.0,
                           eval_metric="auc", tree_method="hist",
                           random_state=70_000+rep, n_jobs=-1)
        xm.fit(X_train, ytp, verbose=False)
        sc = bits_to_atoms(tree_shap(xm, X_eval))
        perm.append({"family":"tree","repeat":rep,
                     "train_auc_permuted":roc_auc_score(ytp, xm.predict_proba(X_train)[:,1]),
                     "test_auc_true":roc_auc_score(y_test, xm.predict_proba(X_test)[:,1]),
                     "pred_sd":float(xm.predict_proba(X_test)[:,1].std()),
                     "final_loss":np.nan,
                     "lift":scored(sc, n=50, base=71_000+rep).mean()})
        pd.DataFrame(perm).to_csv(PERM_PATH, index=False)

perm = pd.DataFrame(perm)
# rows written before the escalation logic carry no flag; derive it from the training fit
if "fitted" not in perm.columns:
    perm["fitted"] = np.nan
perm["fitted"] = perm["fitted"].where(
    perm["fitted"].notna(), perm["train_auc_permuted"] >= FIT_MIN_AUC).astype(bool)
perm.to_csv(PERM_PATH, index=False)

n_ok = perm.groupby("family").fitted.sum().astype(int)
n_all = perm.groupby("family").size()
print()
print("runs that absorbed the randomised labels:")
for fam in n_all.index:
    print(f"  {fam}: {n_ok[fam]}/{n_all[fam]}")
print()
print("all runs")
print(perm.groupby("family")[["train_auc_permuted","test_auc_true","lift"]]
          .agg(["mean","std","min","max"]).round(4).to_string())
print()
print("fitted runs only (the control as defined)")
ok = perm[perm.fitted]
print(ok.groupby("family")[["train_auc_permuted","test_auc_true","lift"]]
        .agg(["mean","std","min","max"]).round(4).to_string())
for fam in perm.family.unique():
    sub = ok[ok.family == fam]
    if len(sub) >= 4:
        r = pearsonr(sub.train_auc_permuted, sub.lift)
        print(f"\n  {fam}: lift vs permuted-training AUC over fitted runs  "
              f"r = {r[0]:+.3f}, p = {r[1]:.3f}, n = {len(sub)}")
    else:
        print(f"\n  {fam}: too few fitted runs ({len(sub)}) for a correlation")
print("\n  A run that collapses to a constant output produces a random atom ranking and a")
print("  lift of one by construction; such runs test nothing and are reported separately.")


runs that absorbed the randomised labels:
  graph: 14/14
  tree: 16/16

all runs
       train_auc_permuted                         test_auc_true                            lift                        
                     mean     std     min     max          mean     std     min     max    mean     std     min     max
family                                                                                                                 
graph              0.8737  0.1411  0.6010  0.9761        0.4916  0.0329  0.4282  0.5502  1.0359  0.0738  0.9008  1.1659
tree               0.9730  0.0017  0.9704  0.9764        0.4938  0.0336  0.4331  0.5428  0.9948  0.0543  0.9133  1.1119

fitted runs only (the control as defined)
       train_auc_permuted                         test_auc_true                            lift                        
                     mean     std     min     max          mean     std     min     max    mean     std     min     max
family                             

## 8. The missing cell: gradient attribution on fingerprints

The comparison between TreeSHAP and Integrated Gradients differs on two axes at once, the
attribution rule and the molecular representation. Circular fingerprint bits are substructure
counters and the alerts are substructures, so the fingerprint arm is aligned with the target
by construction. Applying Integrated Gradients to a network over the same fingerprints, and
distributing the attributions through the same bit-to-atom map, separates the two.

The three readings of this result were fixed in Part 8 before it was run.

In [ ]:
import torch.nn as nn
MLP_PATH = f"{R}/table47_fingerprint_gradient.csv"
if os.path.exists(MLP_PATH):
    mlp_df = pd.read_csv(MLP_PATH); print("loaded from checkpoint")
else:
    class MLP(nn.Module):
        def __init__(self, d=FP_BITS, h1=512, h2=256, p=0.3):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(d,h1), nn.ReLU(), nn.Dropout(p),
                                     nn.Linear(h1,h2), nn.ReLU(), nn.Dropout(p),
                                     nn.Linear(h2,1))
        def forward(self, x): return self.net(x).squeeze(-1)

    Xtr = torch.tensor(X_train, dtype=torch.float, device=DEVICE)
    Ytr = torch.tensor(y_train, dtype=torch.float, device=DEVICE)
    Xva = torch.tensor(X_valid, dtype=torch.float, device=DEVICE)
    Xte = torch.tensor(X_test,  dtype=torch.float, device=DEVICE)
    rows = []
    for sd in range(3):
        set_seed(80_000+sd)
        net = MLP().to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)
        crit = nn.BCEWithLogitsLoss(pos_weight=POS_W)
        best, best_state, waited = -1, None, 0
        for ep in range(200):
            net.train()
            perm_i = torch.randperm(len(Xtr), device=DEVICE)
            for i in range(0, len(Xtr), 128):
                b = perm_i[i:i+128]; opt.zero_grad()
                crit(net(Xtr[b]), Ytr[b]).backward(); opt.step()
            net.eval()
            with torch.no_grad():
                pv = torch.sigmoid(net(Xva)).cpu().numpy()
            a = roc_auc_score(y_valid, pv)
            if a > best:
                best, waited = a, 0
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
            else:
                waited += 1
                if waited >= 20: break
        net.load_state_dict(best_state); net.eval()
        with torch.no_grad():
            pte = torch.sigmoid(net(Xte)).cpu().numpy()
        Xe = torch.tensor(X_eval, dtype=torch.float, device=DEVICE)
        total = torch.zeros_like(Xe)
        for al in torch.linspace(1.0/N_IG_STEPS, 1.0, N_IG_STEPS):
            xa = (Xe*al).detach().requires_grad_(True)
            out = net(xa)
            total += torch.autograd.grad(out.sum(), xa)[0]
        attr = (Xe * (total/N_IG_STEPS)).detach().cpu().numpy()
        for conv, mat in [("absolute", np.abs(attr)), ("signed", attr)]:
            sc = bits_to_atoms(mat)
            rows.append({"seed": 80_000+sd, "convention": conv,
                         "test_roc_auc": roc_auc_score(y_test, pte),
                         "lift": scored(sc, n=100, base=81_000+sd).mean()})
        del net
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    mlp_df = pd.DataFrame(rows); mlp_df.to_csv(MLP_PATH, index=False)

print(mlp_df.round(4).to_string(index=False))
best_conv = mlp_df.groupby("convention").lift.mean().idxmax()
v = mlp_df[mlp_df.convention == best_conv].lift
ig_graph = meta["lift_ig_k30"].mean()
print(f"\n  MLP on fingerprints, Integrated Gradients ({best_conv}): {v.mean():.4f}")
print(f"  graph model, Integrated Gradients          : {ig_graph:.4f}")
print(f"  degree baseline                            : {DEG_LIFT[0.30]:.4f}")
try:
    ts = pd.read_csv(f"{R}/table17_topk_sensitivity.csv")
    ts = ts[(ts.source.str.contains("TreeSHAP")) & (ts.k_fraction == 0.30)].lift.iloc[0]
    print(f"  TreeSHAP on the same fingerprints          : {ts:.4f}")
    print(f"\n  reading: {'the gradient rule' if abs(v.mean()-ig_graph) < abs(v.mean()-ts) else 'the representation'}"
          " accounts for more of the gap")
except Exception:
    pass

loaded from checkpoint
 seed convention  test_roc_auc   lift
80000   absolute        0.7716 1.1206
80000     signed        0.7716 1.4056
80001   absolute        0.7630 1.1723
80001     signed        0.7630 1.4267
80002   absolute        0.7662 1.2144
80002     signed        0.7662 1.4175

  MLP on fingerprints, Integrated Gradients (signed): 1.4166
  graph model, Integrated Gradients          : 1.2655
  degree baseline                            : 1.2136
  TreeSHAP on the same fingerprints          : 1.5422

  reading: the representation accounts for more of the gap


## 9. GNNExplainer under different settings

GNNExplainer was reported as falling below chance. That is a strong claim about a method, and
it was measured at one setting on one model. The mask is optimised, so the setting matters,
and a mask that stays close to uniform produces a ranking that is arbitrary rather than
uninformative. Both are examined here: three settings on five models, with the entropy of the
learned mask recorded alongside.

In [ ]:
GNNEX_PATH = f"{R}/table48_gnnexplainer_settings.csv"
SETTINGS = [{"name":"default",  "epochs":100, "lr":0.01, "node_feat_size":1.0, "node_feat_ent":0.1},
            {"name":"longer",   "epochs":300, "lr":0.01, "node_feat_size":1.0, "node_feat_ent":0.1},
            {"name":"weak_reg", "epochs":100, "lr":0.01, "node_feat_size":0.1, "node_feat_ent":0.01}]

if os.path.exists(GNNEX_PATH):
    gx = pd.read_csv(GNNEX_PATH); print("loaded from checkpoint")
else:
    from torch_geometric.explain import Explainer, GNNExplainer
    from torch_geometric.data import Batch
    def run_gnnexplainer(model, graphs, cfg):
        alg = GNNExplainer(epochs=cfg["epochs"], lr=cfg["lr"])
        try:
            alg.coeffs.update({"node_feat_size": cfg["node_feat_size"],
                               "node_feat_ent":  cfg["node_feat_ent"]})
        except Exception:
            pass
        ex = Explainer(model=model, algorithm=alg, explanation_type="model",
                       node_mask_type="attributes", edge_mask_type="object",
                       model_config=dict(mode="binary_classification",
                                         task_level="graph", return_type="raw"))
        scores, entropies = [], []
        for g in graphs:
            b = Batch.from_data_list([g]).to(DEVICE)
            e = ex(b.x, b.edge_index, edge_attr=b.edge_attr, batch=b.batch)
            nm = e.node_mask.abs().sum(dim=1).detach().cpu().numpy()
            scores.append(nm)
            p = nm / nm.sum() if nm.sum() > 0 else np.full(len(nm), 1/len(nm))
            p = np.clip(p, 1e-12, 1)
            entropies.append(float(-(p*np.log(p)).sum() / np.log(len(nm))) if len(nm) > 1 else np.nan)
        return scores, np.array(entropies)

    rows = []
    for sd in MODEL_SEEDS[:5]:
        m = build_gnn("gin", hp["hidden"], hp["n_layers"], hp["dropout"], hp.get("residual", True))
        m.load_state_dict(torch.load(f"{MODELS}/gin_attr_seed{sd}.pt",
                                     map_location=DEVICE, weights_only=False)["state_dict"])
        m.eval()
        for cfg in SETTINGS:
            t0 = time.time()
            try:
                sc, ent = run_gnnexplainer(m, GRAPHS, cfg)
                rows.append({"seed": sd, "setting": cfg["name"], "epochs": cfg["epochs"],
                             "node_feat_size": cfg["node_feat_size"],
                             "node_feat_ent": cfg["node_feat_ent"],
                             "lift": scored(sc, n=50, base=90_000+sd).mean(),
                             "mask_entropy_mean": float(np.nanmean(ent)),
                             "minutes": (time.time()-t0)/60})
                print(f"  seed {sd} {cfg['name']:9s} lift {rows[-1]['lift']:.4f}  "
                      f"entropy {rows[-1]['mask_entropy_mean']:.3f}  "
                      f"({rows[-1]['minutes']:.1f} min)")
            except Exception as err:
                print(f"  seed {sd} {cfg['name']}: failed ({type(err).__name__}: {err})")
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        pd.DataFrame(rows).to_csv(GNNEX_PATH, index=False)
    gx = pd.DataFrame(rows)

if len(gx):
    print()
    print(gx.groupby("setting")[["lift","mask_entropy_mean"]].agg(["mean","std"]).round(4).to_string())
    print("\n  a mask entropy near 1.0 means the mask is close to uniform and the ranking")
    print("  it induces is arbitrary rather than informative.")

  seed 20000 default: failed (TypeError: GNN.forward() got an unexpected keyword argument 'edge_attr')
  seed 20000 longer: failed (TypeError: GNN.forward() got an unexpected keyword argument 'edge_attr')
  seed 20000 weak_reg: failed (TypeError: GNN.forward() got an unexpected keyword argument 'edge_attr')
  seed 20001 default: failed (TypeError: GNN.forward() got an unexpected keyword argument 'edge_attr')
  seed 20001 longer: failed (TypeError: GNN.forward() got an unexpected keyword argument 'edge_attr')
  seed 20001 weak_reg: failed (TypeError: GNN.forward() got an unexpected keyword argument 'edge_attr')
  seed 20002 default: failed (TypeError: GNN.forward() got an unexpected keyword argument 'edge_attr')
  seed 20002 longer: failed (TypeError: GNN.forward() got an unexpected keyword argument 'edge_attr')
  seed 20002 weak_reg: failed (TypeError: GNN.forward() got an unexpected keyword argument 'edge_attr')
  seed 20003 default: failed (TypeError: GNN.forward() got an unexpected 

## 10. Sensitivity of the untrained distribution to architecture size

The spread of the untrained distribution differed markedly between architectures, from a
standard deviation of 0.059 for GIN to 0.141 for GCN. If that reflects the size of the
network and the scale of its initialisation rather than the message-passing rule, then two
further configurations of the same architecture should reproduce the range. No training is
involved.

In [ ]:
NULLHP_PATH = f"{R}/table49_null_hyperparameters.csv"
if os.path.exists(NULLHP_PATH):
    nh = pd.read_csv(NULLHP_PATH); print("loaded from checkpoint")
else:
    CONFIGS = [{"name":"selected", "hidden":hp["hidden"], "n_layers":hp["n_layers"]},
               {"name":"narrow_shallow", "hidden":32,  "n_layers":2},
               {"name":"wide_deep",      "hidden":256, "n_layers":5}]
    N_DRAWS = 200
    rows = []
    for cfg in CONFIGS:
        vals = np.empty(N_DRAWS)
        t0 = time.time()
        for j in range(N_DRAWS):
            set_seed(95_000+j)
            m = build_gnn("gin", cfg["hidden"], cfg["n_layers"], hp["dropout"],
                          hp.get("residual", True))
            vals[j] = lift(overlap(grad_scores(m, GRAPHS, "ig"), K_FRAC, 95_000+j))
            del m
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        rows.append({"config": cfg["name"], "hidden": cfg["hidden"],
                     "n_layers": cfg["n_layers"], "n_draws": N_DRAWS,
                     "mean": vals.mean(), "sd": vals.std(ddof=1),
                     "p2_5": np.percentile(vals, 2.5), "p97_5": np.percentile(vals, 97.5),
                     "minutes": (time.time()-t0)/60})
        np.save(f"{EXPL}/null_{cfg['name']}.npy", vals)
        print(f"  {cfg['name']:16s} mean {vals.mean():.4f}  sd {vals.std(ddof=1):.4f}  "
              f"({rows[-1]['minutes']:.1f} min)")
        pd.DataFrame(rows).to_csv(NULLHP_PATH, index=False)
    nh = pd.DataFrame(rows)
print()
print(nh.round(4).to_string(index=False))
print("\n  for comparison, the architectures reported earlier: GIN sd 0.059, "
      "GAT 0.125, GCN 0.141")

  selected         mean 1.1132  sd 0.0607  (2.9 min)
  narrow_shallow   mean 1.0942  sd 0.0854  (1.8 min)
  wide_deep        mean 1.1320  sd 0.0550  (3.3 min)

        config  hidden  n_layers  n_draws   mean     sd   p2_5  p97_5  minutes
      selected      64         4      200 1.1132 0.0607 0.9953 1.2247   2.9077
narrow_shallow      32         2      200 1.0942 0.0854 0.8900 1.2382   1.7853
     wide_deep     256         5      200 1.1320 0.0550 1.0206 1.2320   3.2968

  for comparison, the architectures reported earlier: GIN sd 0.059, GAT 0.125, GCN 0.141


## 11. Applicability domain and conformal coverage for the descriptor model

The strongest classical model is absent from three tables. Its predictions are already
stored, so no fitting is required.

In [ ]:
from scipy import stats
z = np.load(f"{R}/predictions_primary.npz")
ze = np.load(f"{R}/predictions_extra.npz")
sim = np.load(f"{R}/ad_nn_similarity.npy")
p_desc = ze["p_descriptor"]

rows = []
for thr in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    inside = sim >= thr
    if inside.sum() < 30 or (~inside).sum() < 30: continue
    a_in  = roc_auc_score(y_test[inside],  p_desc[inside])
    a_out = roc_auc_score(y_test[~inside], p_desc[~inside])
    rows.append({"model":"XGBoost (RDKit-2D)","threshold":thr,
                 "n_inside":int(inside.sum()),"n_outside":int((~inside).sum()),
                 "auc_inside":a_in,"auc_outside":a_out,"delta":a_out-a_in})
ad = pd.DataFrame(rows)
ad.to_csv(f"{R}/table50_descriptor_ad.csv", index=False)
print(ad.round(4).to_string(index=False))

ALPHA = 0.10
def mondrian(p_valid, y_valid, p_test, alpha=ALPHA):
    cal = {c: np.sort(np.where(c == 1, 1-p_valid[y_valid == c], p_valid[y_valid == c]))
           for c in (0, 1)}
    inc = {}
    for c in (0, 1):
        s = 1-p_test if c == 1 else p_test
        n = len(cal[c])
        pv = np.array([(np.sum(cal[c] >= si)+1)/(n+1) for si in s])
        inc[c] = pv > alpha
    return inc[1], inc[0]

try:
    pv_desc = ze["p_descriptor_valid"]
    ip, ineg = mondrian(pv_desc, y_valid, p_desc)
    cov = np.where(y_test == 1, ip, ineg)
    size = ip.astype(int) + ineg.astype(int)
    k, N = int(cov.sum()), len(cov)
    print(f"\nconformal, class-conditional: coverage {cov.mean():.4f} "
          f"[{stats.beta.ppf(0.025,k,N-k+1):.4f}, {stats.beta.ppf(0.975,k+1,N-k):.4f}], "
          f"mean set size {size.mean():.3f}, uncertain {100*(size==2).mean():.1f}%")
    pd.DataFrame([{"model":"XGBoost (RDKit-2D)","scheme":"mondrian","coverage":cov.mean(),
                   "avg_set_size":size.mean(),"uncertain_rate":(size==2).mean()}]).to_csv(
        f"{R}/table51_descriptor_conformal.csv", index=False)
except KeyError:
    print("\nvalidation predictions for the descriptor model were not stored; "
          "conformal coverage for it is reported from table27 instead")

             model  threshold  n_inside  n_outside  auc_inside  auc_outside   delta
XGBoost (RDKit-2D)       0.25       667         59      0.8125       0.6355 -0.1770
XGBoost (RDKit-2D)       0.30       599        127      0.8192       0.6782 -0.1410
XGBoost (RDKit-2D)       0.35       504        222      0.8286       0.6844 -0.1442
XGBoost (RDKit-2D)       0.40       445        281      0.8354       0.6983 -0.1371
XGBoost (RDKit-2D)       0.50       296        430      0.8318       0.7638 -0.0679

validation predictions for the descriptor model were not stored; conformal coverage for it is reported from table27 instead


## 12. Verification of the precision-recall computation

The graph model leads on ROC-AUC and trails on PR-AUC, which is unusual enough to check
rather than interpret. Average precision is recomputed here from the stored probabilities and
compared with the values in the benchmark table.

In [ ]:
chk = []
for m in ["XGBoost","RandomForest","GCN","GIN","GAT"]:
    p = z[f"test_{m}"]
    chk.append({"model": m, "roc_auc": roc_auc_score(y_test, p),
                "pr_auc_recomputed": average_precision_score(y_test, p),
                "positive_rate": float(y_test.mean()),
                "pr_auc_of_random": float(y_test.mean())})
chk = pd.DataFrame(chk)
chk.to_csv(f"{R}/table52_pr_auc_check.csv", index=False)
print(chk.round(4).to_string(index=False))
print("\n  a random ranking scores PR-AUC equal to the positive rate; every model is above it.")
print("  ROC-AUC and PR-AUC are both rank based, so calibration cannot explain a divergence;")
print("  a larger gap means the model orders the top of the ranking relatively less well.")

       model  roc_auc  pr_auc_recomputed  positive_rate  pr_auc_of_random
     XGBoost   0.7824             0.7690         0.4628            0.4628
RandomForest   0.7994             0.7856         0.4628            0.4628
         GCN   0.7877             0.7445         0.4628            0.4628
         GIN   0.8133             0.7663         0.4628            0.4628
         GAT   0.7977             0.7799         0.4628            0.4628

  a random ranking scores PR-AUC equal to the positive rate; every model is above it.
  ROC-AUC and PR-AUC are both rank based, so calibration cannot explain a divergence;
  a larger gap means the model orders the top of the ranking relatively less well.


## 13. Summary

In [ ]:
print("=== attribution across twenty models, k = 0.30 ===")
for meth, lab in [("ig","Integrated Gradients"),("gradinput","Gradient x Input"),("saliency","Saliency")]:
    v = meta[f"lift_{meth}_k30"].values
    print(f"  {lab:22s} {v.mean():.4f} +/- {v.std(ddof=1):.4f}   "
          f"exceeds degree in {100*(v>DEG_LIFT[0.30]).mean():.0f}% of models")
print(f"  {'degree baseline':22s} {DEG_LIFT[0.30]:.4f}")

print("\n=== agreement versus performance ===")
print(perf[["method","pearson_r","pearson_p"]].round(4).to_string(index=False))

print("\n=== permutation control ===")
print(perm.groupby("family")[["train_auc_permuted","test_auc_true","lift"]].mean().round(4).to_string())

print("\n=== missing cell ===")
print(mlp_df.groupby("convention")[["test_roc_auc","lift"]].mean().round(4).to_string())

print("\nWritten to", R)
for f in sorted(glob.glob(f"{R}/table4*.csv")) + sorted(glob.glob(f"{R}/table5*.csv")):
    print("  -", os.path.basename(f))
print(f"\n{len(glob.glob(f'{MODELS}/gin_attr_seed*.pt'))} model weights and "
      f"{len(glob.glob(f'{EXPL}/scores_*_seed*.npz'))} score files retained")

import shutil
from google.colab import files
shutil.make_archive("/content/part9_results", "zip", R)
files.download("/content/part9_results.zip")

=== attribution across twenty models, k = 0.30 ===
  Integrated Gradients   1.2655 +/- 0.0916   exceeds degree in 75% of models
  Gradient x Input       1.1887 +/- 0.0911   exceeds degree in 40% of models
  Saliency               1.2694 +/- 0.1010   exceeds degree in 75% of models
  degree baseline        1.2136

=== agreement versus performance ===
   method  pearson_r  pearson_p
       ig     0.2557     0.2766
gradinput    -0.0040     0.9866
 saliency    -0.2076     0.3797

=== permutation control ===
        train_auc_permuted  test_auc_true    lift
family                                           
graph               0.8737         0.4916  1.0359
tree                0.9730         0.4938  0.9948

=== missing cell ===
            test_roc_auc    lift
convention                      
absolute          0.7669  1.1691
signed            0.7669  1.4166

Written to /content/drive/MyDrive/ames_pipeline/results
  - table40_superiority.csv
  - table41_partition_dependence.csv
  - table42_mod

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>